# Cricket Over Runs Prediction — Linear vs Gradient Boosting vs Polynomial Regression

Predicts how many runs will be scored in one T20 over, using 3 models.

The answer is a number (not yes/no), so this is a **regression** problem.

**How to run:** Run each cell top to bottom (Shift+Enter). Nothing needs to be uploaded.

## Phase 1: Build the dataset
This cell creates the data, so there is no file to upload.

The numbers follow real T20 patterns: death overs score more, fallen wickets slow the
scoring down, a better batter scores more and a weaker bowler concedes more.

*Note: this is simulated practice data, not a real scorecard.*

In [ ]:
import pandas as pd, numpy as np

rng = np.random.default_rng(42)                          # same data every run
N = 2000                                                 # 2000 overs

over = rng.integers(1, 21, N)                            # over 1 to 20
wkt  = np.clip(rng.poisson(over * 0.3), 0, 9)            # wickets so far
bat  = np.round(rng.normal(32, 8, N).clip(12, 55), 1)    # batter's average
eco  = np.round(rng.normal(8, 1.2, N).clip(5, 12), 2)    # bowler's economy

runs = 5.75 + 0.3*over - 0.6*wkt + 0.18*(bat-32) + 1.3*(eco-8) + 4*(over >= 16)

df = pd.DataFrame({'over_number': over, 'wickets_fallen': wkt, 'batsman_avg': bat,
                   'bowler_econ': eco,
                   'runs_in_over': np.clip(rng.poisson(np.clip(runs, 1, 25)), 0, 35)})

df.loc[rng.choice(N, 100, replace=False), 'batsman_avg'] = 0    # fake missing values
df.loc[rng.choice(N, 100, replace=False), 'bowler_econ'] = 0

print(df.shape)
df.head()

## Phase 2: Inspect the dataset
A batting average of 0 or an economy of 0 is impossible — those are missing values hiding
as `0`. `df.info()` won't catch them, so we count them ourselves.

In [ ]:
df.info()

print("\nHidden missing values (zeros):")
print((df[['batsman_avg', 'bowler_econ']] == 0).sum())

## Phase 3: Clean the data
Replace the impossible 0s with each column's median (median is used because outliers
don't shift it).

The two lines must stay in this order: the zeros become `NaN` **first**, so that the
median is calculated without them dragging it down.

In [ ]:
for col in ['batsman_avg', 'bowler_econ']:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

print((df[['batsman_avg', 'bowler_econ']] == 0).sum())

## Phase 4: Split into train and test sets
`X` = the 4 inputs. `y` = runs scored. 80% trains the models, 20% tests them on unseen overs.

Scores used: **MAE** = how many runs off on average (lower is better),
**R²** = how much the model explains, 1.0 is perfect (higher is better).

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df.drop('runs_in_over', axis=1), df['runs_in_over']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training overs:", len(X_train), "| Testing overs:", len(X_test))

## Phase 5: Train all 3 models

| Model | Idea |
|---|---|
| **Linear Regression** | One straight-line weight per input — the baseline |
| **Gradient Boosting** | Many small decision trees, each fixing the last one's mistakes |
| **Polynomial Regression** | Linear Regression, but the inputs are also squared and multiplied together |

The three are stored in a dictionary and trained in one loop, so adding a fourth model
would take a single extra line.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, r2_score

models = {'Linear Regression': LinearRegression(),
          'Gradient Boosting': GradientBoostingRegressor(random_state=42),
          'Polynomial Regression': make_pipeline(PolynomialFeatures(2), LinearRegression())}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = model.predict(X_test)
    print(f"{name:<22} MAE: {mean_absolute_error(y_test, results[name]):.2f} runs"
          f"   R2: {r2_score(y_test, results[name]):.3f}")

## Phase 6: Compare the 3 models visually
Each dot is one test over: the actual runs across, the predicted runs up.
**The closer the dots sit to the red line, the better the model.**

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, pred) in zip(axes, results.items()):
    ax.scatter(y_test, pred, alpha=0.3)
    ax.plot([0, 25], [0, 25], 'r--')
    ax.set_title(f'{name}\nMAE: {mean_absolute_error(y_test, pred):.2f}'
                 f' | R2: {r2_score(y_test, pred):.3f}')
    ax.set_xlabel('Actual runs')
    ax.set_ylabel('Predicted runs')

plt.tight_layout()
plt.show()

## Conclusion

| Model | MAE | R² |
|---|---|---|
| **Gradient Boosting** | **2.37 runs** | **0.542** |
| **Polynomial Regression** | **2.37 runs** | **0.541** |
| Linear Regression | 2.52 runs | 0.491 |

- **Gradient Boosting and Polynomial Regression finish level.** The gap between them is
  0.001 — far too small to call one the winner on this split.
- Both clearly beat plain **Linear Regression**, which can only add its inputs up in a
  straight line.
- The reason both win: the death-over jump only pays off *when wickets are still in hand*.
  `PolynomialFeatures(2)` captures that by multiplying the inputs together, and the decision
  trees capture it with if-then splits. A plain straight line cannot bend that way.

### Which one is actually better?

One 20% test split is a small sample, so the two being 0.001 apart proves nothing. Running
**5-fold cross-validation** over the whole dataset separates them:

| Model | R² (5-fold) |
|---|---|
| **Polynomial Regression** | **0.523** |
| Gradient Boosting | 0.515 |
| Linear Regression | 0.484 |

**Polynomial Regression is slightly ahead — and it is one line of code against 100 decision
trees.** For the same result, the simpler model is the better engineering choice.

R² near 0.5 is a fair result here, not a failure: about half of an over depends on the match
situation, the rest is luck — one six swings the over by 6 runs and no model can predict that.

**Main lesson:** the biggest model is not automatically the best one. Giving a simple model
better features matched a much heavier algorithm.